# Project 53 — ML **With Twitter**

Same Final Gun Violence step order as `P53_ML_No_Twitter.ipynb`, plus a **Twitter join** first.

| Item | Value |
|------|--------|
| Socio input | `hex_53_MASTER.csv` |
| Twitter input | `twitter_hex_features.csv` keyed by `GRID_ID` |
| Outputs | `outputs/with_twitter/WithTwitter_Preprocessed.csv`, `outputs/with_twitter/WithTwitter_VIF_Clean.csv`, `outputs/with_twitter/WithTwitter_Predictions.csv` |

### How Twitter is added (Final / Quack method)
1. Load hex socioeconomic master (`GRID_ID`).
2. Left-join hex-level Twitter features on `GRID_ID`.
3. Typical fields: tweet volume, likes/retweets, sentiment shares, unique users, topic/threat scores (`t_m1_*`, `t_m2_*`, `t0_*`).
4. Fill missing Twitter values with 0 (hexes with no tweets).
5. Run the same Final ML pipeline on the merged table.

### Important — Project 53 today
Existing Quack-lab Twitter CSVs use **Final** GRID_IDs (e.g. `BRD-1200`, ~2,022 hexes, 4 cities).
Project 53 hexes use a **new** national tessellation (e.g. `CUB-1173`) — **GRID overlap = 0**.
Until `twitter_hex_features.csv` is built for Project 53 grids, this notebook will stop after the join check (it will not invent Twitter columns).


## Step 0 — Twitter join check

In [ ]:
import matplotlib
matplotlib.use("Agg")

from io import BytesIO
from IPython.display import Image, display

def show_plot_in_notebook():
    """Display current matplotlib figure in Jupyter even with Agg backend."""
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    display(Image(data=buf.getvalue()))


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

SOCIO = Path("input/hex_53_MASTER.csv")
TWITTER_CANDIDATES = [
    Path("input/twitter_hex_features.csv"),
    Path("../data/twitter_hex_features.csv"),
]

print("=" * 80)
print("STEP 0: LOAD SOCIO + TWITTER JOIN")
print("=" * 80)

df_socio = pd.read_csv(SOCIO)
print(f"Loaded socio master: {SOCIO} -> {df_socio.shape[0]:,} rows x {df_socio.shape[1]} cols")

twitter_path = next((p for p in TWITTER_CANDIDATES if p.exists()), None)
TWITTER_READY = False
df = df_socio.copy()

if twitter_path is None:
    print("\nTWITTER STATUS: FILE NOT FOUND")
    print("  Looked for:")
    for p in TWITTER_CANDIDATES:
        print(f"   - {p.resolve()}")
    print("\nProvide twitter_hex_features.csv with GRID_ID + numeric Twitter columns.")
    print("GRID_ID must match Project 53 hexes (e.g. CUB-1173), not Final (BRD-1200).")
    print("Reference schema (different grids — do not join as-is):")
    print(r"  C:\Users\svina\Desktop\Quack lab\Twitter\Raw_data\Grid_Twitter_Analysis_Final.csv")
else:
    tw = pd.read_csv(twitter_path)
    print(f"Found Twitter file: {twitter_path} -> {tw.shape[0]:,} rows x {tw.shape[1]} cols")
    if "GRID_ID" not in tw.columns:
        raise ValueError("twitter_hex_features.csv must include GRID_ID")
    overlap = len(set(df_socio["GRID_ID"]) & set(tw["GRID_ID"]))
    print(f"GRID_ID overlap with socio master: {overlap:,} / {len(df_socio):,}")
    if overlap == 0:
        print("\nTWITTER STATUS: ZERO GRID OVERLAP — cannot join.")
    else:
        drop_meta = [c for c in ["Latitude", "Longitude", "City_name", "incident_place", "incident_date"] if c in tw.columns]
        tw_feat = tw.drop(columns=drop_meta, errors="ignore")
        dup = [c for c in tw_feat.columns if c != "GRID_ID" and c in df_socio.columns]
        tw_feat = tw_feat.drop(columns=dup, errors="ignore")
        df = df_socio.merge(tw_feat, on="GRID_ID", how="left")
        tw_cols = [c for c in tw_feat.columns if c != "GRID_ID"]
        for c in tw_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        TWITTER_READY = True
        print(f"TWITTER STATUS: JOINED {len(tw_cols)} Twitter columns")
        print(f"Merged shape: {df.shape[0]:,} x {df.shape[1]}")

if not TWITTER_READY:
    print("\n" + "=" * 80)
    print("STOPPING: With-Twitter pipeline needs a valid Project 53 Twitter hex table.")
    print("Use P53_ML_No_Twitter.ipynb for socioeconomic-only ML.")
    print("=" * 80)
    raise SystemExit("Twitter hex features not available for Project 53 GRID_IDs")

print("\nTwitter join OK — continuing Final-style ML steps on merged table...")



## How Twitter was joined to the hex master

There were **two joins**, not one.

### Join A — tweets → hex (done before this notebook)

Each tweet has lat/lon. We asked: **which hex polygon contains this point?** That hex’s `GRID_ID` is written on the tweet.

Then we **summarized per hex** (tweet count, % anger, % grief, etc.).  
That file is `input/twitter_hex_features.csv`.

### Join B — Twitter summary → master (this Step 0)

We match Twitter rows to the socio master on **`GRID_ID`**.  
Keep **all** hexes. If a hex has no tweets, fill Twitter columns with **0**.

---

### Tiny example — Join A

3 hexes: `CUB-1`, `CUB-2`, `CUB-3`  
5 tweets:

| Tweet | Where it lands | Emotion |
|-------|----------------|---------|
| 1 | CUB-1 | anger |
| 2 | CUB-1 | grief |
| 3 | CUB-2 | anger |
| 4 | CUB-2 | anger |
| 5 | CUB-2 | fear |

A tweet in Uvalde falls **outside** all 53-city hexes → **no GRID_ID**, dropped.

Summarize per `GRID_ID`:

| GRID_ID | tw_tweet_count | tw_pct_anger | tw_pct_grief | tw_pct_fear |
|---------|----------------|--------------|--------------|-------------|
| CUB-1 | 2 | 50 | 50 | 0 |
| CUB-2 | 3 | 67 | 0 | 33 |

`CUB-3` has **zero tweets**, so it is **not** in the Twitter file yet.

---

### Tiny example — Join B (Step 0)

Socio master (every hex):

| GRID_ID | Total_Gun_Violence | SNAP |
|---------|--------------------|------|
| CUB-1 | 4 | 200 |
| CUB-2 | 1 | 80 |
| CUB-3 | 0 | 50 |

Left join on `GRID_ID`:

| GRID_ID | Total_Gun_Violence | SNAP | tw_tweet_count | tw_pct_anger | tw_pct_grief |
|---------|--------------------|------|----------------|--------------|--------------|
| CUB-1 | 4 | 200 | **2** | **50** | **50** |
| CUB-2 | 1 | 80 | **3** | **67** | **0** |
| CUB-3 | 0 | 50 | **0** | **0** | **0** |

`CUB-3` stays in the table. Twitter is just empty.

---

### What this notebook printed (real data)

| Piece | Value |
|--------|--------|
| Master hexes | 14,415 × 51 |
| Twitter file | 12 rows × 32 cols |
| GRID_ID overlap | **12 / 14,415** |
| Twitter columns added | 31 `tw_*` |
| Merged table | 14,415 × 82 |

- **858** tweets landed inside a P53 hex  
- **766** tweets were outside (Uvalde, Highland Park, etc.)  
- Biggest hex: Nashville **`CAH-553`** (~831 tweets)

---

### One sentence

Tweet lat/lon → which hex? → add up emotions in that hex → attach those numbers to the same `GRID_ID` on the master; everyone else gets 0.

After this, Steps 1–28 are the same as the No-Twitter notebook, on this 82-column table.

---

# PHASE 1: DATA PREPARATION

---

## Step 1: Load Dataset and Initial Exploration

### 🎯 Purpose
We start by loading the raw dataset and performing initial exploratory analysis. Think of this like a doctor doing an initial patient examination before treatment - we need to understand what we're working with.

### 📚 Why This Step Matters
Before we can train any machine learning model, we need to understand our data:
- **Know the data shape**: How many grids and features do we have?
- **Identify problems early**: Are there missing values? Unusual patterns?
- **Establish baseline**: What does our target variable (gun violence) look like?
- **Quality check**: Make sure the data is clean and ready for modeling

### What We'll Check
1. **Dataset dimensions** (rows × columns) - Total grids and features
2. **Missing values** across all features - Empty cells?
3. **Data types** for each column - Numbers? Text? Categories?
4. **Basic statistics** (mean, median, min, max, std) - Range and spread
5. **Target variable distribution** (Total_Gun_Violence) - Violence prevalence

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("="*80)
print("STEP 1: LOAD DATASET AND INITIAL EXPLORATION")
print("="*80)

In [ ]:
# Socio+Twitter table already built in Step 0 as `df`
print("="*80)
print("STEP 1: CONFIRM MERGED DATASET")
print("="*80)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nAll fields ({len(df.columns)}):")
for idx, column_name in enumerate(df.columns, start=1):
    print(f"  {idx:2d}. {column_name}")


In [ ]:
# Display first few rows
print("\n📋 First 5 rows of the dataset:")
display(df.head())

In [ ]:
# Check for missing values
print("\n🔍 Missing Values Analysis:")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

cols_with_missing = missing_df[missing_df['Missing_Count'] > 0]
if len(cols_with_missing) > 0:
    print(f"\n⚠️  Found {len(cols_with_missing)} columns with missing values:")
    display(cols_with_missing)
else:
    print("\n✓ No missing values found in dataset")

In [ ]:
# Analyze target variable
print("\n🎯 Target Variable Analysis: Total_Gun_Violence")
print("\nDistribution Statistics:")
print(df['Total_Gun_Violence'].describe())

print(f"\nZero violence grids: {(df['Total_Gun_Violence'] == 0).sum():,} "
      f"({(df['Total_Gun_Violence'] == 0).sum()/len(df)*100:.1f}%)")
print(f"Skewness: {df['Total_Gun_Violence'].skew():.2f}")
print(f"Kurtosis: {df['Total_Gun_Violence'].kurt():.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Total_Gun_Violence'], bins=50, color='crimson', alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Total Gun Violence')
axes[0].set_xlabel('Total Gun Violence Incidents')
axes[0].grid(alpha=0.3)

axes[1].boxplot(df['Total_Gun_Violence'])
axes[1].set_title('Box Plot')
axes[1].grid(alpha=0.3)
plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 1 COMPLETE")
print("="*80)


## Step 2: Remove Redundant and "Cheat" Features

### 🎯 Purpose
Clean the dataset by removing features that shouldn't be used for predictions. Think of this like cleaning your desk - remove irrelevant items to focus on what matters.

### 📚 Why We Remove Features?

**1. "Cheat" Features (Data Leakage) - ⚠️ CRITICAL**
- These are PARTS of the target we're predicting!
- Example: We want to predict "Total_Gun_Violence" but using "Children_Killed" as a feature is cheating!
- Why? Because "Children_Killed" IS PART OF total violence - already counted in our target
- **Danger if we don't remove**: Model gets 99% accuracy on training data but 30% on new data (overfitting disaster!)
- **Real impact**: Police deploy resources based on wrong signals

**2. Geometric/Administrative Features - Zero Predictive Power**
- Shape_Length, Shape_Area = Just grid hexagon geometry (irrelevant to violence)
- QA_Flag = Just an administrative marker (data quality flag, not predictive)
- These tell us nothing about whether an area will have gun violence

**3. Redundant Geographic Features**
- City_name is already captured by grid coordinates
- Redundant information wastes model's learning capacity

### Features to Remove by Category
- **Cheat**: Accidental_Death, Accidental_Injuries, Children_Injured, Children_Killed, Mass_Shootings, Teens_injured, Teens_Killed, Bars
- **Geometric**: Shape_Length, Shape_Area
- **Admin**: QA_Flag
- **Redundant**: City_name

In [ ]:
print("="*80)
print("STEP 2: REMOVE REDUNDANT AND 'CHEAT' FEATURES")
print("="*80)

# Define features to drop
cheat_features = [
    'Accidental_Death', 'Accidental_Injuries', 'Children_Injured',
    'Children_Killed', 'Mass_Shootings', 'Teens_injured', 'Teens_Killed', 'Bars'
]
geometric_features = ['Shape_Length', 'Shape_Area']
admin_features = [
    'QA_Flag', 'aggregationMethod', 'HasData', 'ORIGINAL_OID',
    'sourceCountry', 'apportionmentConfidence', 'populationToPolygonSizeRating'
]
geographic_features = ['City_name', 'State']

features_to_drop = cheat_features + geometric_features + admin_features + geographic_features
existing_drops = [feature for feature in features_to_drop if feature in df.columns]

print(f"\nRemoving {len(existing_drops)} features...")
for idx, feature_name in enumerate(existing_drops, start=1):
    print(f"  {idx:2d}. {feature_name}")

df_clean = df.drop(columns=existing_drops)

print(f"\n✓ New shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print("\n" + "="*80)
print("✓ STEP 2 COMPLETE")
print("="*80)

## Step 3: Convert Counts to Percentages

### 🎯 Purpose
Convert absolute violence counts into percentages of the population. This is like comparing "10 car accidents" - meaningless until we know if it's a town of 500 people or 500,000 people!

### 📚 Why Convert to Percentages?

**Problem with Raw Counts:**
- A dense urban area like downtown Chicago has 1,000 gun violence incidents
- A rural area has 50 incidents  
- Does this mean downtown is 20x more dangerous? Not necessarily if it has 20x more people!
- Raw counts are **not comparable across areas with different populations**

**Why Percentages Work:**
- **10 incidents per 1,000 people (1%)** is much higher violence rate than **10 incidents per 10,000 people (0.1%)**
- Percentages normalize by population, making fair comparisons across cities
- Model learns **density and concentration patterns**, not just raw numbers

**Technical Impact:**
- Violence_Rate (%) = (Total_Gun_Violence / Total_Population) × 100
- Makes small towns and big cities directly comparable
- Captures true risk level of neighborhoods regardless of size
- Same process for all demographic features: Black%, Hispanic%, White%, etc.

In [ ]:
print("="*80)
print("STEP 3: DEMOGRAPHIC PERCENTAGES")
print("="*80)

df_pct = df_clean.copy()
total_pop = df_pct['Total_Population'].replace(0, np.nan)

created_percentage_features = []

# Race percentages
if 'Black_Population' in df_pct.columns:
    df_pct['Black_Percentage'] = (df_pct['Black_Population'] / total_pop * 100).fillna(0)
    created_percentage_features.append('Black_Percentage')
if 'White_Population' in df_pct.columns:
    df_pct['White_Percentage'] = (df_pct['White_Population'] / total_pop * 100).fillna(0)
    created_percentage_features.append('White_Percentage')

# Poverty rate
if 'Population_Below_Poverty_Level' in df_pct.columns:
    df_pct['Poverty_Rate_Pct'] = (df_pct['Population_Below_Poverty_Level'] / total_pop * 100).fillna(0).clip(upper=100)
    created_percentage_features.append('Poverty_Rate_Pct')

# Education percentages (normalize by educational-attainment base)
if 'Total_Educational_Attainment_Base' in df_pct.columns:
    edu_base = df_pct['Total_Educational_Attainment_Base'].replace(0, np.nan)
    if 'Education_Less_Than_Ninth_Grade' in df_pct.columns:
        df_pct['Education_Less_Than_9th_Pct'] = (df_pct['Education_Less_Than_Ninth_Grade'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_Less_Than_9th_Pct')
    if 'Education_Max_Have_High_School_Diploma' in df_pct.columns:
        df_pct['Education_HS_Diploma_Pct'] = (df_pct['Education_Max_Have_High_School_Diploma'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_HS_Diploma_Pct')
    if 'Education_Have_Bachelors_Degree' in df_pct.columns:
        df_pct['Education_Bachelors_Pct'] = (df_pct['Education_Have_Bachelors_Degree'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_Bachelors_Pct')

# Remove raw count fields corresponding to percentage features
raw_count_features_to_remove = [
    'Black_Population',
    'White_Population',
    'Population_Below_Poverty_Level',
    'Education_Less_Than_Ninth_Grade',
    'Education_Max_Have_High_School_Diploma',
    'Education_Have_Bachelors_Degree',
    'Total_Educational_Attainment_Base'
]
existing_raw_count_features = [col for col in raw_count_features_to_remove if col in df_pct.columns]
if existing_raw_count_features:
    df_pct = df_pct.drop(columns=existing_raw_count_features)

new_pct_features = [col for col in df_pct.columns if col.endswith('_Percentage') or col.endswith('_Pct')]

print(f"\n✓ Created {len(created_percentage_features)} percentage features")
for idx, feature_name in enumerate(created_percentage_features, start=1):
    print(f"  {idx:2d}. {feature_name}")

print(f"\n✓ Removed {len(existing_raw_count_features)} related raw count columns")
for idx, feature_name in enumerate(existing_raw_count_features, start=1):
    print(f"  {idx:2d}. {feature_name}")

if existing_raw_count_features:
    print("\nℹ️ Why removed these raw columns:")
    print("  - They are raw-count counterparts of the new percentage features.")
    print("  - Keeping both versions adds redundant signal and can bias models toward scale effects.")
    print("  - Using normalized percentages improves comparability across hexagons.")

print("\n" + "="*80)
print("✓ STEP 3 COMPLETE")
print("="*80)

## Step 4: Min-Max Scaling

### 🎯 Purpose
Scale numeric predictors to a common range [0, 1] so models compare features fairly instead of favoring variables with larger raw units.

### 📚 Why This Step Matters
- **What it is**: Min-Max scaling transforms each feature using `(x - min) / (max - min)`.
- **Why we use it**: Logistic Regression and distance/split behaviors can become unstable when feature ranges differ a lot.
- **Benefit**: Better numerical stability, faster convergence, and coefficients/importances that are easier to compare across features.

### ✅ Output
A scaled modeling table with `GRID_ID` preserved and `Total_Gun_Violence` unchanged as the target variable.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Fill remaining NaNs so MinMaxScaler does not fail
_preserve = ['GRID_ID', 'Total_Gun_Violence']
for _col in df_pct.columns:
    if _col not in _preserve:
        df_pct[_col] = pd.to_numeric(df_pct[_col], errors='coerce').fillna(0)

print("="*80)
print("STEP 4: MIN-MAX SCALING")
print("="*80)

# Preserve identifiers and target
preserve_cols = ['GRID_ID', 'Total_Gun_Violence']
df_preserved = df_pct[preserve_cols].copy()

# Scale numeric features
numeric_cols = df_pct.select_dtypes(include=[np.number]).columns.tolist()
cols_to_scale = [col for col in numeric_cols if col not in preserve_cols]

scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled_values = scaler.fit_transform(df_pct[cols_to_scale])
df_scaled = pd.DataFrame(df_scaled_values, columns=cols_to_scale, index=df_pct.index)

# Merge back
df_scaled = pd.concat([df_preserved, df_scaled], axis=1)

# Reorder: GRID_ID first, target last
other_cols = [col for col in df_scaled.columns if col not in preserve_cols]
df_scaled = df_scaled[['GRID_ID'] + other_cols + ['Total_Gun_Violence']]

print(f"\n✓ Scaled {len(cols_to_scale)} features to [0, 1] range")
print("\n" + "="*80)
print("✓ STEP 4 COMPLETE")
print("="*80)

## Step 5: Save Preprocessed Dataset

### 🎯 Purpose
Persist the cleaned and scaled dataset to disk so downstream steps use a reproducible input.

### 📚 Why This Step Matters
- **What it is**: A checkpoint export after feature cleaning, percentage conversion, and scaling.
- **Why we use it**: Prevents recomputing earlier steps every run and supports auditing/reuse.
- **Benefit**: Reproducible experiments and easier debugging if later steps change.

In [ ]:
print("="*80)
print("STEP 5: SAVE PREPROCESSED DATASET")
print("="*80)

from pathlib import Path
Path('outputs/with_twitter').mkdir(parents=True, exist_ok=True)
output_file = 'outputs/with_twitter/WithTwitter_Preprocessed.csv'
df_scaled.to_csv(output_file, index=False)

print(f"\n✓ Saved: {output_file}")
print(f"  Shape: {df_scaled.shape}")
print("\n" + "="*80)
print("✓ STEP 5 COMPLETE")
print("="*80)

---

# PHASE 2: MULTICOLLINEARITY ANALYSIS

---

## Step 6-7: VIF Analysis - Initial Calculation and Iterative Pruning

### 🎯 Purpose
Detect and remove multicollinearity so predictors carry non-duplicative signal.

### 📚 What is Multicollinearity?
Multicollinearity happens when multiple features encode nearly the same information. This inflates variance of coefficients and makes model interpretation unstable.

### 📚 What is VIF and Why We Use It
- **VIF (Variance Inflation Factor)** measures how strongly one feature is explained by the others.
- **Rule used here**: iteratively remove the worst feature until all remaining features have `VIF ≤ 5`.
- **Benefit**: More stable coefficients, better generalization, and cleaner interpretation of feature effects.

### ✅ Output
A pruned feature set that reduces redundancy before model training.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("="*80)
print("STEP 6-7: VIF ANALYSIS")
print("="*80)

# Prepare data
df_vif_input = df_scaled.drop(columns=['GRID_ID', 'Total_Gun_Violence'], errors='ignore')

def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["Feature"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(len(df.columns))]
    return vif_data.sort_values('VIF', ascending=False)

# Iterative pruning
df_vif_clean = df_vif_input.copy()
removed_features = []
iteration = 0

print("\n⚙️  Starting iterative VIF pruning...\n")

while True:
    iteration += 1
    vif_current = calculate_vif(df_vif_clean)
    max_vif = vif_current['VIF'].max()
    
    if max_vif <= 5:
        print(f"\n✓ Convergence: All VIF ≤ 5 after {iteration-1} iterations")
        break
    
    worst_feature = vif_current.iloc[0]['Feature']
    worst_vif = vif_current.iloc[0]['VIF']
    
    print(f"Iteration {iteration}: Removing {worst_feature} (VIF={worst_vif:.2f})")
    
    df_vif_clean = df_vif_clean.drop(columns=[worst_feature])
    removed_features.append(worst_feature)
    
    if iteration > 100:
        print("\n⚠️  Exceeded 100 iterations, stopping...")
        break

vif_final = calculate_vif(df_vif_clean)
print(f"\n✓ Final features: {len(df_vif_clean.columns)}")
print(f"✓ Max VIF: {vif_final['VIF'].max():.2f}")
print(f"✓ Removed: {len(removed_features)} features")

print("\n" + "="*80)
print("✓ STEP 6-7 COMPLETE")
print("="*80)

## Step 8: Pairwise Correlation Check

### 🎯 Purpose
Apply a second redundancy filter after VIF by checking highly correlated feature pairs.

### 📚 Why This Step Matters
- **What it is**: Correlation analysis over remaining features (threshold `|r| ≥ 0.75`).
- **Why we use it**: Even after VIF pruning, some feature pairs can still be strongly duplicated.
- **How we decide drops**: For each high-correlation pair, keep the feature with stronger target correlation and remove the other.
- **Benefit**: Smaller, less redundant feature space with improved model robustness.

In [ ]:
print("="*80)
print("STEP 8: PAIRWISE CORRELATION CHECK")
print("="*80)

corr_matrix = df_vif_clean.corr()
threshold = 0.75

# Find high correlation pairs
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = []

for column in upper_tri.columns:
    high_corr_features = upper_tri[column][abs(upper_tri[column]) >= threshold].index.tolist()
    for feature in high_corr_features:
        high_corr_pairs.append({'Feature_1': column, 'Feature_2': feature, 
                                 'Correlation': upper_tri.loc[feature, column]})

print(f"\n🔍 Found {len(high_corr_pairs)} high-correlation pairs (|r| ≥ {threshold})")

features_to_drop_corr = []
if len(high_corr_pairs) > 0:
    # Add target temporarily
    df_with_target = df_vif_clean.copy()
    df_with_target['Total_Gun_Violence'] = df_scaled['Total_Gun_Violence']
    target_corr = df_with_target.corr()['Total_Gun_Violence'].drop('Total_Gun_Violence')
    
    for pair in high_corr_pairs:
        feat1, feat2 = pair['Feature_1'], pair['Feature_2']
        corr1, corr2 = abs(target_corr[feat1]), abs(target_corr[feat2])
        drop_feat = feat2 if corr1 >= corr2 else feat1
        
        if drop_feat not in features_to_drop_corr:
            features_to_drop_corr.append(drop_feat)
            print(f"  ❌ Dropping: {drop_feat}")
    
    df_vif_clean = df_vif_clean.drop(columns=features_to_drop_corr)

print(f"\n✓ Removed {len(features_to_drop_corr)} additional features")
print(f"✓ Final feature count: {len(df_vif_clean.columns)}")

print("\n" + "="*80)
print("✓ STEP 8 COMPLETE")
print("="*80)

## Step 9: Save VIF-Clean Dataset

### 🎯 Purpose
Export the post-collinearity feature matrix that will be used for EDA and model training.

### 📚 Why This Step Matters
- **What it is**: A checkpoint after VIF and correlation pruning.
- **Why we use it**: Freezes the final predictor set so model results are reproducible.
- **Benefit**: Ensures the exact same cleaned feature space is used across all models.

In [ ]:
print("="*80)
print("STEP 9: SAVE VIF-CLEAN DATASET")
print("="*80)

df_final_clean = pd.concat([
    df_scaled['GRID_ID'],
    df_vif_clean,
    df_scaled['Total_Gun_Violence']
], axis=1)

output_file_clean = 'outputs/with_twitter/WithTwitter_VIF_Clean.csv'
df_final_clean.to_csv(output_file_clean, index=False)

print(f"\n✓ Saved: {output_file_clean}")
print(f"  Shape: {df_final_clean.shape}")
print(f"  Features for modeling: {df_final_clean.shape[1] - 2}")

print("\n" + "="*80)
print("✓ STEP 9 COMPLETE")
print("="*80)

---

# PHASE 3: EXPLORATORY DATA ANALYSIS

---

## Step 10-11: Target Distribution and Feature Correlations

### 🎯 Purpose
Understand target behavior and rank candidate predictors before fitting models.

### 📚 Why This Step Matters
- **Step 10 (Target distribution)**: Shows imbalance, skew, and spread in violence outcomes.
- **Step 11 (Feature correlation)**: Screens for variables with strongest linear relationship to the target.
- **Benefit**: Informs model expectations, evaluation strategy, and interpretability priorities.

### 📝 Important Note: Correlation "Top Predictors" ≠ ML Feature Importance
- In this step, "top predictors" means **top correlated features** based on Pearson correlation with `Total_Gun_Violence`.
- This is **EDA** (a pre-model diagnostic), not model training and not SHAP/feature-importance output.
- We print top values here to understand data signal early, while final model importance is analyzed later in Step 19-21.

### ⚠️ Leakage Caution
- Correlation inspection is safe for understanding, but if you use it for feature selection, compute it on the **training split only** to avoid test-data leakage.

In [ ]:
print("="*80)
print("STEP 10: TARGET DISTRIBUTION ANALYSIS")
print("="*80)

print("\n🎯 Target Variable: Total_Gun_Violence")
print(df_final_clean['Total_Gun_Violence'].describe())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_final_clean['Total_Gun_Violence'], bins=30, color='crimson', alpha=0.7)
axes[0].set_title('Target Distribution')
axes[0].grid(alpha=0.3)

axes[1].boxplot(df_final_clean['Total_Gun_Violence'])
axes[1].set_title('Box Plot')
axes[1].grid(alpha=0.3)
plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 10 COMPLETE")
print("="*80)


In [ ]:
print("="*80)
print("STEP 11: FEATURE CORRELATION WITH TARGET")
print("="*80)

feature_cols = [col for col in df_final_clean.columns if col not in ['GRID_ID', 'Total_Gun_Violence']]
target_correlations = df_final_clean[feature_cols + ['Total_Gun_Violence']].corr()['Total_Gun_Violence'].drop('Total_Gun_Violence')
target_corr_sorted = target_correlations.abs().sort_values(ascending=False)

print("\nTop 10 features by correlation:")
for i, (feat, corr) in enumerate(zip(target_corr_sorted.index[:10], target_correlations[target_corr_sorted.index[:10]]), 1):
    print(f"  {i}. {feat}: {corr:.3f}")

# Visualize
top_n = 15
top_features = target_correlations.sort_values(ascending=False).head(top_n)
plt.figure(figsize=(12, 6))
plt.barh(range(len(top_features)), top_features.values, color='forestgreen', alpha=0.7)
plt.yticks(range(len(top_features)), top_features.index, fontsize=9)
plt.xlabel('Correlation with Gun Violence')
plt.title(f'Top {top_n} Predictors')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 11 COMPLETE")
print("="*80)


---

# PHASE 4: MODEL TRAINING & EVALUATION

---

## Step 12: Train-Test Split

### 🎯 Purpose
Create independent training and test sets to evaluate generalization on unseen hexagons.

### 📚 Why This Step Matters
- **What it is**: Stratified split keeps class balance similar between train and test.
- **Why we use it**: Prevents optimistic bias from evaluating on data the model already saw.
- **Benefit**: Reliable estimate of real-world predictive performance.

In [ ]:
from sklearn.model_selection import train_test_split

print("="*80)
print("STEP 12: TRAIN-TEST SPLIT")
print("="*80)

# Separate features and target
X = df_final_clean.drop(columns=['GRID_ID', 'Total_Gun_Violence'])
y_binary = (df_final_clean['Total_Gun_Violence'] > 0).astype(int)
grid_ids = df_final_clean['GRID_ID']

print(f"\nDataset: {len(X):,} samples, {X.shape[1]} features")
print(f"Class 0 (No violence): {(y_binary == 0).sum():,} ({(y_binary == 0).sum()/len(y_binary)*100:.1f}%)")
print(f"Class 1 (Any violence): {(y_binary == 1).sum():,} ({(y_binary == 1).sum()/len(y_binary)*100:.1f}%)")

# Split
X_train, X_test, y_train, y_test, grid_train, grid_test = train_test_split(
    X, y_binary, grid_ids,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print(f"\nTraining: {len(X_train):,} samples")
print(f"Test: {len(X_test):,} samples")
print(f"Training class balance (violence=1): {y_train.mean():.3f}")
print(f"Test class balance (violence=1): {y_test.mean():.3f}")

print("\n" + "="*80)
print("✓ STEP 12 COMPLETE")
print("="*80)

## Step 13-18: Train ML Models

### 🎯 Purpose
Fit multiple classifiers and compare them on the same holdout data.

### 📚 Why This Step Matters
- **What it is**: Baseline + tree-ensemble models trained under consistent split/features.
- **Why we use it**: No single algorithm is best for all geospatial risk problems.
- **Benefit**: Model benchmarking lets us select the strongest performer (by F1 and ROC-AUC) for deployment-oriented scoring.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

print("="*80)
print("STEP 13-18: TRAINING ML MODELS")
print("="*80)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, 
                              scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]), eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, class_weight='balanced', verbose=-1)
}

# Train and evaluate
results = []

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred_test = model.predict(X_test)
    y_pred_proba_test = model.predict_proba(X_test)[:, 1]
    
    test_metrics = {
        'Accuracy': accuracy_score(y_test, y_pred_test),
        'Precision': precision_score(y_test, y_pred_test, zero_division=0),
        'Recall': recall_score(y_test, y_pred_test),
        'F1': f1_score(y_test, y_pred_test),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba_test)
    }
    
    print(f"\n✓ Training complete ({train_time:.2f}s)")
    print(f"\nTest Set Performance:")
    print(f"  Accuracy:  {test_metrics['Accuracy']:.4f}")
    print(f"  Precision: {test_metrics['Precision']:.4f}")
    print(f"  Recall:    {test_metrics['Recall']:.4f}")
    print(f"  F1 Score:  {test_metrics['F1']:.4f}")
    print(f"  ROC-AUC:   {test_metrics['ROC-AUC']:.4f}")
    
    results.append({
        'Model': name,
        'Train_Time': train_time,
        'Test_Accuracy': test_metrics['Accuracy'],
        'Test_Precision': test_metrics['Precision'],
        'Test_Recall': test_metrics['Recall'],
        'Test_F1': test_metrics['F1'],
        'Test_ROC_AUC': test_metrics['ROC-AUC']
    })

# Create comparison table
results_df = pd.DataFrame(results)

print(f"\n\n{'='*80}")
print("MODEL COMPARISON - TEST SET PERFORMANCE")
print(f"{'='*80}\n")
display(results_df)

# Identify best model
best_model_idx = results_df['Test_F1'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1 = results_df.loc[best_model_idx, 'Test_F1']

print(f"\n🏆 BEST MODEL: {best_model_name} (F1 = {best_f1:.4f})")
best_model = models[best_model_name]

print("\n" + "="*80)
print("✓ STEP 13-18 COMPLETE")
print("="*80)

## Step 19-21: Feature Importance Analysis

### 🎯 Purpose
Explain model behavior, not just model accuracy, using global and local interpretation tools.

### 📚 Why These Steps Matter
- **Step 19 (Feature importance)**: Aggregates which variables are most influential across models.
- **Step 20 (SHAP)**: Uses game-theoretic attributions to quantify each feature’s impact on predictions.
- **Step 21 (PDP)**: Shows how predicted risk changes as one feature varies while others are averaged.

### ✅ Benefit
These explainability steps support policy decisions by showing *why* hexagons are predicted as high risk.

In [ ]:
import shap
from sklearn.inspection import PartialDependenceDisplay

print("="*80)
print("STEP 19: FEATURE IMPORTANCE")
print("="*80)

# Get feature importance from models
importance_data = []

for name, model in models.items():
    if hasattr(model, 'feature_importances_'):
        for feat, imp in zip(X.columns, model.feature_importances_):
            importance_data.append({'Model': name, 'Feature': feat, 'Importance': imp})
    elif hasattr(model, 'coef_'):
        for feat, imp in zip(X.columns, np.abs(model.coef_[0])):
            importance_data.append({'Model': name, 'Feature': feat, 'Importance': imp})

importance_df = pd.DataFrame(importance_data)
avg_importance = importance_df.groupby('Feature')['Importance'].mean().sort_values(ascending=False)

print("\nTop 20 Features:")
print(avg_importance.head(20))

# Visualize
plt.figure(figsize=(12, 8))
top_20 = avg_importance.head(20)
plt.barh(range(len(top_20)), top_20.values, color='teal', alpha=0.7)
plt.yticks(range(len(top_20)), top_20.index, fontsize=9)
plt.xlabel('Average Importance')
plt.title('Top 20 Features by Average Importance')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 19 COMPLETE")
print("="*80)


In [ ]:
print("="*80)
print("STEP 20: SHAP ANALYSIS (optional)")
print("="*80)
try:
    import shap
    from io import BytesIO
    from IPython.display import Image, display

    print(f"Computing SHAP for {best_model_name}...")
    if best_model_name == "Logistic Regression":
        explainer = shap.LinearExplainer(best_model, X_train)
    else:
        explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test, plot_type="bar", show=False, max_display=20)
    plt.title(f"SHAP: {best_model_name}")
    plt.tight_layout()

    # Show in Jupyter even when matplotlib backend is Agg
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close()
    buf.seek(0)
    display(Image(data=buf.getvalue()))
except Exception as e:
    print(f"Skipping SHAP ({type(e).__name__}: {e})")
print("STEP 20 COMPLETE")


In [ ]:
print("="*80)
print("STEP 21: PARTIAL DEPENDENCE PLOTS")
print("="*80)

from io import BytesIO
from IPython.display import Image, display

# Get top 6 features from SHAP
shap_importance = np.abs(shap_values).mean(axis=0)
top_6_indices = np.argsort(shap_importance)[-6:][::-1]
top_6_features = X.columns[top_6_indices].tolist()

print("\nGenerating partial dependence plots for top 6 features:")
for feat in top_6_features:
    print(f"  - {feat}")

fig, ax = plt.subplots(figsize=(16, 10))
PartialDependenceDisplay.from_estimator(
    best_model, X_test, top_6_features,
    ax=ax, n_cols=3, grid_resolution=50
)
fig.suptitle(f'Partial Dependence Plots: {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()

buf = BytesIO()
plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
plt.close()
buf.seek(0)
display(Image(data=buf.getvalue()))

print("\n" + "="*80)
print("✓ STEP 21 COMPLETE")
print("="*80)


## Step 22-24: Classification Metrics

### 🎯 Purpose
Evaluate the selected model(s) comprehensively using threshold-based and ranking metrics.

### 📚 Why This Step Matters
- **Step 22**: Precision/Recall/F1 and class report quantify error trade-offs.
- **Step 23**: Confusion matrices make false positives/false negatives explicit.
- **Step 24**: ROC and Precision-Recall curves compare discrimination quality across all thresholds.
- **Benefit**: Supports transparent threshold and deployment decisions.

### 🧠 Step 24 Notes: ROC vs Precision-Recall (PR)
- **ROC Curve (Receiver Operating Characteristic)** plots **True Positive Rate (Recall)** vs **False Positive Rate** across thresholds.
- **ROC-AUC** summarizes this curve: higher AUC means better class separation overall.
- ROC is useful for broad discrimination checks, but can look optimistic when the dataset is imbalanced.

- **Precision-Recall Curve** plots **Precision** vs **Recall** across thresholds.
- PR focuses on positive-class performance (violence hotspots), which is usually the operational priority.
- In imbalanced data, PR is often more informative than ROC for understanding false-alarm vs missed-hotspot trade-offs.

### ✅ How to Interpret in This Project
- Prefer models/thresholds that keep **high recall** without collapsing **precision**.
- Use ROC-AUC for overall ranking, and PR behavior for deployment decisions in hotspot detection.
- Final threshold selection is then refined in Step 26 using F1/precision/recall trade-offs.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve

print("="*80)
print("STEP 22: CLASSIFICATION METRICS")
print("="*80)

y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

print(f"\nModel: {best_model_name}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No Violence', 'Violence'], digits=4))

print("\n" + "="*80)
print("✓ STEP 22 COMPLETE")
print("="*80)

In [ ]:
print("="*80)
print("STEP 23: CONFUSION MATRICES")
print("="*80)

from io import BytesIO
from IPython.display import Image, display

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')

for idx, (name, model) in enumerate(models.items()):
    if idx >= 6:
        break
    ax = axes[idx // 3, idx % 3]
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['No Violence', 'Violence'],
                yticklabels=['No Violence', 'Violence'])
    ax.set_title(f'{name}\n(F1={f1_score(y_test, y_pred):.4f})', fontweight='bold')

if len(models) < 6:
    axes[1, 2].axis('off')

plt.tight_layout()
buf = BytesIO()
plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
plt.close()
buf.seek(0)
display(Image(data=buf.getvalue()))

print("\\n" + "="*80)
print("✓ STEP 23 COMPLETE")
print("="*80)



In [ ]:
print("="*80)
print("STEP 24: ROC & PRECISION-RECALL CURVES")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    axes[1].plot(recall, precision, label=name, linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)

plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 24 COMPLETE")
print("="*80)


## Step 25: Hotspot Detection

### 🎯 Purpose
Score every hexagon and convert probabilities into actionable hotspot labels.

### 📚 Why This Step Matters
- **What it is**: Per-hexagon probability scoring on the full dataset.
- **Why we use it**: Creates citywide risk surfaces for intervention planning.
- **Benefit**: Produces both best-model probability and explicit logistic-regression probability for GIS mapping and comparison.

In [ ]:
print("="*80)
print("STEP 25: HOTSPOT DETECTION")
print("="*80)

# Get predictions for entire dataset
X_full = df_final_clean.drop(columns=['GRID_ID', 'Total_Gun_Violence'])
y_full_pred_proba_best = best_model.predict_proba(X_full)[:, 1]

y_full_pred_proba_lr = models['Logistic Regression'].predict_proba(X_full)[:, 1]

# Define hotspot threshold (95th percentile)
hotspot_threshold = np.percentile(y_full_pred_proba_best, 95)
hotspot_flag = (y_full_pred_proba_best >= hotspot_threshold).astype(int)
num_hotspots = hotspot_flag.sum()

print(f"\nHotspot threshold (95th percentile): {hotspot_threshold:.4f}")
print(f"Identified {num_hotspots:,} hotspot grids ({num_hotspots/len(X_full)*100:.1f}%)")

# Create composite risk score
risk_score = (
    0.7 * (y_full_pred_proba_best / y_full_pred_proba_best.max()) +
    0.3 * (df_final_clean['Total_Gun_Violence'] / df_final_clean['Total_Gun_Violence'].max())
)

# Create results dataframe
results_for_export = df_final_clean[['GRID_ID', 'Total_Gun_Violence']].copy()
results_for_export['Predicted_Probability_Best_Model'] = y_full_pred_proba_best
results_for_export['Logistic_Regression_Probability'] = y_full_pred_proba_lr
results_for_export['Hotspot_Flag'] = hotspot_flag
results_for_export['Risk_Score'] = risk_score

print(f"\n📊 Summary:")
print(f"  Hotspot grids: {num_hotspots:,}")
print(f"  Avg risk (hotspots): {risk_score[hotspot_flag == 1].mean():.4f}")
print(f"  Avg risk (non-hotspots): {risk_score[hotspot_flag == 0].mean():.4f}")

print("\n" + "="*80)
print("✓ STEP 25 COMPLETE")
print("="*80)

## Step 26: Classification Threshold Tuning

### 🎯 Purpose
Select an operating threshold that matches project priorities (balance between catching risk vs false alarms).

### 📚 Why This Step Matters
- **What it is**: Search over probability thresholds and compute precision/recall/F1 at each value.
- **Why we use it**: Default `0.50` is often suboptimal for imbalanced safety datasets.
- **Benefit**: Data-driven threshold selection aligned with operational goals.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("="*80)
print("STEP 26: CLASSIFICATION THRESHOLD TUNING")
print("="*80)

# Predicted probabilities from best model
y_scores = best_model.predict_proba(X_test)[:, 1]

# Search thresholds to maximize F1
threshold_grid = np.arange(0.10, 0.91, 0.01)
metrics_by_threshold = []

for threshold in threshold_grid:
    y_pred_threshold = (y_scores >= threshold).astype(int)
    metrics_by_threshold.append({
        'Threshold': threshold,
        'Precision': precision_score(y_test, y_pred_threshold, zero_division=0),
        'Recall': recall_score(y_test, y_pred_threshold, zero_division=0),
        'F1': f1_score(y_test, y_pred_threshold, zero_division=0)
    })

threshold_df = pd.DataFrame(metrics_by_threshold)
best_threshold_idx = threshold_df['F1'].idxmax()
selected_threshold = threshold_df.loc[best_threshold_idx, 'Threshold']

# Compare default vs optimized threshold
y_pred_default = (y_scores >= 0.50).astype(int)
y_pred_optimized = (y_scores >= selected_threshold).astype(int)

print(f"\nSelected threshold (max F1): {selected_threshold:.2f}")
print("\nDefault threshold (0.50):")
print(f"  Precision: {precision_score(y_test, y_pred_default, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_default, zero_division=0):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_default, zero_division=0):.4f}")

print("\nOptimized threshold:")
print(f"  Precision: {precision_score(y_test, y_pred_optimized, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_optimized, zero_division=0):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_optimized, zero_division=0):.4f}")

# Plot precision/recall/F1 vs threshold
plt.figure(figsize=(12, 6))
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], label='Precision', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], label='Recall', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['F1'], label='F1 Score', linewidth=2)
plt.axvline(selected_threshold, color='black', linestyle='--', linewidth=2, label=f'Best threshold={selected_threshold:.2f}')
plt.xlabel('Classification Threshold')
plt.ylabel('Metric Value')
plt.title(f'Threshold Tuning: {best_model_name}', fontweight='bold')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
show_plot_in_notebook()

print("\n" + "="*80)
print("✓ STEP 26 COMPLETE")
print("="*80)


## Step 27-28: Export Results and Executive Summary

### 🎯 Purpose
Package modeling outputs for ArcGIS and communicate final findings clearly.

### 📚 Why This Step Matters
- **Step 27 (Export)**: Writes per-hexagon prediction outputs to CSV for map symbology/gradients.
- **Step 28 (Summary)**: Consolidates preprocessing, model performance, explainability, and risk results.
- **Benefit**: Delivers a decision-ready artifact and a concise technical narrative for stakeholders.

In [ ]:
print("="*80)
print("STEP 27: EXPORT PREDICTIONS")
print("="*80)

output_predictions = 'outputs/with_twitter/WithTwitter_Predictions.csv'
results_for_export.to_csv(output_predictions, index=False)

print(f"\n✓ Predictions exported: {output_predictions}")
print(f"  Rows: {len(results_for_export):,}")
print("  Columns: GRID_ID, Total_Gun_Violence, Predicted_Probability_Best_Model, Logistic_Regression_Probability, Hotspot_Flag, Risk_Score")

print("\n" + "="*80)
print("✓ STEP 27 COMPLETE")
print("="*80)

In [ ]:
print("="*80)
print("STEP 28: EXECUTIVE SUMMARY")
print("="*80)

print("\n" + "="*80)
print("GUN VIOLENCE PREDICTION PROJECT - FINAL SUMMARY")
print("="*80)

print(f"\n📊 DATASET")
print(f"  ├─ Total grids: {len(df_final_clean):,}")
print(f"  ├─ Features (after VIF): {X.shape[1]}")
print(f"  └─ Target: Binary classification ({(y_binary == 0).mean()*100:.1f}% zeros)")

print(f"\n🔧 PREPROCESSING")
print(f"  ├─ Removed {len(existing_drops)} non-predictive features")
print(f"  ├─ Removed {len(removed_features)} multicollinear features (VIF > 5)")
print(f"  ├─ Removed {len(features_to_drop_corr)} high-correlation features (|r| ≥ 0.75)")
print(f"  └─ Applied Min-Max scaling (0-1 range)")

print(f"\n🤖 MODELS TRAINED")
print(f"  ├─ Logistic Regression (baseline)")
print(f"  ├─ Random Forest")
print(f"  ├─ Gradient Boosting")
print(f"  ├─ XGBoost")
print(f"  └─ LightGBM")

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"  ├─ Test Accuracy:  {results_df.loc[best_model_idx, 'Test_Accuracy']:.4f}")
print(f"  ├─ Test Precision: {results_df.loc[best_model_idx, 'Test_Precision']:.4f}")
print(f"  ├─ Test Recall:    {results_df.loc[best_model_idx, 'Test_Recall']:.4f}")
print(f"  ├─ Test F1 Score:  {results_df.loc[best_model_idx, 'Test_F1']:.4f}")
print(f"  └─ Test ROC-AUC:   {results_df.loc[best_model_idx, 'Test_ROC_AUC']:.4f}")

print(f"\n🎚️ CLASSIFICATION THRESHOLD")
print(f"  └─ F1-optimized threshold: {selected_threshold:.2f}")

print(f"\n🎯 TOP 10 FEATURES (Average Importance)")
for i, (feat, imp) in enumerate(zip(avg_importance.head(10).index, avg_importance.head(10).values), 1):
    print(f"  {i:2d}. {feat}: {imp:.4f}")

print(f"\n🔥 HOTSPOT DETECTION")
print(f"  ├─ Threshold: 95th percentile (prob ≥ {hotspot_threshold:.4f})")
print(f"  ├─ Hotspot grids: {num_hotspots:,} ({num_hotspots/len(X_full)*100:.1f}%)")
print(f"  └─ Avg risk (hotspots): {risk_score[hotspot_flag == 1].mean():.4f}")

print(f"\n📁 OUTPUT FILES")
print(f"  ├─ outputs/with_twitter/WithTwitter_Preprocessed.csv")
print(f"  ├─ outputs/with_twitter/WithTwitter_VIF_Clean.csv")
print(f"  └─ outputs/with_twitter/WithTwitter_Predictions.csv")

print(f"\n⚠️  LIMITATIONS")
print(f"  ├─ Highly imbalanced target")
print(f"  ├─ Model trained on historical data")
print(f"  ├─ Geographic patterns may not generalize")
print(f"  └─ Socioeconomic factors are correlates, not causes")

print(f"\n💡 RECOMMENDATIONS")
print(f"  ├─ Focus resources on identified hotspot grids")
print(f"  ├─ Monitor top predictive features (e.g., {avg_importance.index[0]})")
print(f"  ├─ Retrain model quarterly with new data")
print(f"  └─ Validate with local law enforcement expertise")

print("\n" + "="*80)
print("✓ STEP 28 COMPLETE")
print("="*80)

print("\n" + "="*80)
print("✅ ALL STEPS COMPLETE - PIPELINE FINISHED SUCCESSFULLY")
print("="*80)
print("\n🎉 Gun Violence Prediction Pipeline Complete!")
print("   All outputs saved and ready for analysis.")
print("="*80)